In [ ]:
# Authentication: run `earthengine authenticate` once and set the environment
# variable EE_PROJECT to your own Google Cloud project before executing.
import os
# -*- coding: utf-8 -*-
#"""
#2025_07_pc1_scene_trends_temporal_gap_fill.ipynb
#Corrigido: MK (S, Z, significant) + estatísticas (mean/median/stdDev) por município
#Saída: projects/YOUR_PROJECT_2/assets/RSEI_TRENDS_PC1_GAP
#"""

# ============================================================
# 0) Setup
# ============================================================
import ee
import re
import sys

# Auth (Colab)
ee.Authenticate()
ee.Initialize(project=os.environ.get('EE_PROJECT'))

# (Opcional em Colab)
from google.colab import auth
auth.authenticate_user()



In [ ]:
# ============================================================
# 1) Parâmetros
# ============================================================
asset_indices = 'projects/nexgenmap/MapBiomas2/LANDSAT/DEGRADACAO/mosaics-ndfia'   # deve conter a banda PC1
asset_city    = 'projects/YOUR_PROJECT/assets/CAP_JOIN2'                         # FC com NM_MUN
asset_out_root = 'projects/YOUR_PROJECT_2/assets/RSEI_TRENDS_PC1_GAP'

# Lista (exemplo) – pode expandir
MUNICIPIOS = [
    'Salvador'
    # 'Porto Velho', 'Rio Branco', ...
]

# Threshold de significância (bicaudal)
P_THRESHOLD = 0.05  # mude para 0.025 se quiser mais conservador

# ============================================================
# 2) Utilitários
# ============================================================
def sanitize_id(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]', '_', name)

def ensure_image_collection(ic_asset_id: str):
    try:
        info = ee.data.getAsset(ic_asset_id)
        if info and info.get('type') == 'IMAGE_COLLECTION':
            return
        elif info:
            print(f"[AVISO] Asset existe mas não é ImageCollection: {ic_asset_id} (type={info.get('type')}).")
            return
    except Exception:
        pass
    try:
        ee.data.createAsset({'type': ee.data.ASSET_TYPE_IMAGE_COLL}, ic_asset_id)
        print(f"[OK] ImageCollection criada: {ic_asset_id}")
    except Exception as e:
        print(f"[ERRO] Falha ao criar ImageCollection {ic_asset_id}: {e}")

# ============================================================
# 3) Gap-fill temporal (PC1)
# ============================================================
def temporal_gapfill_ic_bidirectional(ic, band_name='PC1'):
    """
    Preenche pixels mascarados usando info temporal (t+1 e t-1).
    Retorna coleção em ordem ASC e preserva system:time_start.
    """
    ic = ic.select([band_name])

    ic_asc  = ic.sort('system:time_start')
    size    = ic_asc.size()
    ic_desc = ic_asc.sort('system:time_start', False)
    list_desc = ic_desc.toList(size)

    init_acc = ee.Dictionary({'prev': ee.Image(0).updateMask(ee.Image(0)),
                              'list': ee.List([])})

    def _step_backward(el, acc):
        acc  = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        filled = img.unmask(prev)
        filled = filled.copyProperties(img, img.propertyNames())
        return ee.Dictionary({'prev': filled,
                              'list': ee.List(acc.get('list')).add(filled)})

    back_dict = ee.Dictionary(ee.List(list_desc).iterate(_step_backward, init_acc))
    list_back_asc = ee.List(back_dict.get('list')).reverse()
    ic_back_asc = ee.ImageCollection.fromImages(list_back_asc)

    list_fwd = ic_back_asc.toList(size)
    init_acc_fwd = ee.Dictionary({'prev': ee.Image(0).updateMask(ee.Image(0)),
                                  'list': ee.List([])})

    def _step_forward(el, acc):
        acc  = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        filled = img.unmask(prev)
        filled = filled.copyProperties(img, img.propertyNames())
        return ee.Dictionary({'prev': filled,
                              'list': ee.List(acc.get('list')).add(filled)})

    fwd_dict = ee.Dictionary(ee.List(list_fwd).iterate(_step_forward, init_acc_fwd))
    list_filled = ee.List(fwd_dict.get('list'))
    ic_filled = ee.ImageCollection.fromImages(list_filled)

    return ic_filled.sort('system:time_start').select([band_name])

# ============================================================
# 4) Normalização 0–1 (mesmo min/max para toda a coleção, na região)
# ============================================================
def normalize_ic_01(ic, band_name, region, scale=30):
    minImgT = ic.select(band_name).min()
    maxImgT = ic.select(band_name).max()

    minDict = minImgT.reduceRegion(
        reducer=ee.Reducer.min(), geometry=region, scale=scale,
        maxPixels=1e13, bestEffort=True
    )
    maxDict = maxImgT.reduceRegion(
        reducer=ee.Reducer.max(), geometry=region, scale=scale,
        maxPixels=1e13, bestEffort=True
    )

    minv = ee.Number(ee.Dictionary(minDict).get(band_name, 0))
    maxv = ee.Number(ee.Dictionary(maxDict).get(band_name, 1))
    rng  = maxv.subtract(minv)
    safe_rng = ee.Number(ee.Algorithms.If(rng.gt(0), rng, 1))

    minImg   = ee.Image.constant(minv)
    safeRImg = ee.Image.constant(safe_rng)

    def _map_norm(img):
        img  = ee.Image(img)
        norm = img.select(band_name).subtract(minImg).divide(safeRImg).rename(band_name)
        # preservar só o timestamp (necessário pro join temporal)
        norm = norm.copyProperties(img, ['system:time_start'])
        return norm

    ic_norm = ic.map(_map_norm)
    return {'collection': ic_norm, 'min': minv, 'max': maxv}

# ============================================================
# 5) MK por município (corrigido)
# ============================================================
def mk_for_municipio(municipio: str):
    try:
        # Geometria
        fc_city = ee.FeatureCollection(asset_city).filter(ee.Filter.eq('NM_MUN', municipio))
        geom_city = fc_city.geometry()

        # Coleção filtrada por município
        ic_raw = ee.ImageCollection(asset_indices) \
                    .filter(ee.Filter.eq('NM_MUN', municipio)) \
                    .sort('system:time_start')

        n_info = ic_raw.size().getInfo()
        if not n_info or n_info < 2:
            return {'municipio': municipio, 'status': 'pulado', 'msg': f'{n_info} imagem(ns); precisa de >= 2.'}

        bands0 = ee.Image(ic_raw.first()).bandNames().getInfo()
        if 'PC1' not in bands0:
            return {'municipio': municipio, 'status': 'erro', 'msg': "Banda 'PC1' não encontrada na coleção."}

        # Seleciona PC1 e clip
        ic_pc1 = ic_raw.select(['PC1']).map(lambda img: ee.Image(img).clip(geom_city))

        # Gap-fill temporal
        ic_pc1_filled = temporal_gapfill_ic_bidirectional(ic_pc1, 'PC1')

        # Normalização 0–1
        norm_res = normalize_ic_01(ic_pc1_filled, 'PC1', geom_city, scale=30)
        ic = ee.ImageCollection(norm_res['collection'])  # coleção normalizada

        # ---------- Mann–Kendall: S ----------
        time_filter   = ee.Filter.lessThan(leftField='system:time_start', rightField='system:time_start')
        save_all_join = ee.Join.saveAll(matchesKey='after', ordering='system:time_start', ascending=True)
        joined        = save_all_join.apply(ic, ic, time_filter)

        def sign(i, j):
            i = ee.Image(i)
            j = ee.Image(j)
            # sinal de (j - i), ignorando empates
            return j.neq(i).multiply(j.subtract(i).clamp(-1, 1)).int()

        # soma dos sinais (S)
        kendall = ee.ImageCollection(
            joined.map(lambda current:
                ee.ImageCollection.fromImages(ee.List(ee.Image(current).get('after'))).map(
                    lambda image: sign(ee.Image(current), ee.Image(image)).unmask(0)
                )
            ).flatten()
        ).reduce(ee.Reducer.sum()).rename('S').clip(geom_city)

        # Variância (sem correção por 'ties')
        n_num = ee.Number(n_info)
        kendall_variance = n_num.multiply(n_num.subtract(1)) \
                                .multiply(n_num.multiply(2).add(5)) \
                                .divide(18)

        # ---------- Z (CORRIGIDO) ----------
        S = kendall.select('S').toFloat()
        sqrt_var = kendall_variance.sqrt()

        Z = ee.Image(0).where(S.gt(0), S.subtract(1).divide(sqrt_var)) \
                       .where(S.lt(0), S.add(1).divide(sqrt_var)) \
                       .rename('Z') \
                       .clip(geom_city)

        # ---------- p bicaudal e significância ----------
        # CDF normal padrão usando erf
        absZ = Z.abs()
        cdf = ee.Image(0.5).multiply(ee.Image(1).add(absZ.divide(ee.Image(2).sqrt()).erf()))
        p_two_tailed = ee.Image(2).multiply(ee.Image(1).subtract(cdf)).rename('p').clip(geom_city)
        significant = p_two_tailed.lte(P_THRESHOLD).rename('significant').toFloat().clip(geom_city)  # 0/1 em float

        # ---------- Estatísticas temporais ----------
        mean_img   = ic.reduce(ee.Reducer.mean()).rename(['mean']).toFloat().clip(geom_city)
        median_img = ic.reduce(ee.Reducer.median()).rename(['median']).toFloat().clip(geom_city)
        std_img    = ic.reduce(ee.Reducer.stdDev()).rename(['stdDev']).toFloat().clip(geom_city)

        # Empacota as bandas
        out_img = ee.Image.cat([
            significant,   # 0/1 float
            Z,             # Z
            S,             # S
            mean_img,
            median_img,
            std_img
        ]).rename(['significant','Z','S','mean','median','stdDev']).clip(geom_city)

        # Propriedades úteis
        src_img   = ee.Image(ic_raw.first())
        start_date = ee.Date(ee.Image(ic.first()).get('system:time_start'))
        end_date   = ee.Date(ee.Image(ic.sort('system:time_start', False).first()).get('system:time_start'))
        m_pairs    = n_num.multiply(n_num.subtract(1)).divide(2)

        out_img = out_img.copyProperties(source=src_img, properties=src_img.propertyNames()) \
                         .set({
                             'NM_MUN': municipio,
                             'mk:first_band': 'PC1',
                             'mk:n_images': n_num,
                             'mk:p_threshold': P_THRESHOLD,
                             'mk:variance': kendall_variance,
                             'mk:m_pairs': m_pairs,
                             'mk:start': start_date.format('YYYY-MM-dd'),
                             'mk:end':   end_date.format('YYYY-MM-dd'),
                             'mk:asset_indices': asset_indices,
                             'mk:asset_city': asset_city,
                             'mk:description': 'MK (S, Z, significant) + estatísticas (mean/median/stdDev) da PC1 (norm 0–1) por município'
                         })

        # Export
        muni_safe   = sanitize_id(municipio)
        asset_id_out = f"{asset_out_root}/MK_{muni_safe}"

        task = ee.batch.Export.image.toAsset(
            image=out_img,
            description=f"MK_{muni_safe}",
            assetId=asset_id_out,
            region=geom_city,
            scale=30,
            maxPixels=1e13
        )

        return {'municipio': municipio, 'status': 'ok', 'asset_out': asset_id_out, 'task': task}

    except Exception as e:
        return {'municipio': municipio, 'status': 'erro', 'msg': str(e)}



In [ ]:
# ============================================================
# 6) Garantir coleção de saída
# ============================================================
ensure_image_collection(asset_out_root)

# ============================================================
# 7) Loop principal (cria e inicia tasks)
# ============================================================
results = []
for m in MUNICIPIOS:
    info = mk_for_municipio(m)
    results.append(info)
    if info.get('status') == 'ok':
        print(f"[OK] Task criada: {info['municipio']} -> {info['asset_out']}")
        if 'task' in info and info['task'] is not None:
            info['task'].start()
        else:
            print(f"[ERRO] Task não encontrada no resultado para {info['municipio']}")
    else:
        print(f"[{info.get('status','erro').upper()}] {m}: {info.get('msg','')}")

# ============================================================
# 8) Resumo
# ============================================================
ok_count   = sum(1 for r in results if r.get('status') == 'ok')
skip_count = sum(1 for r in results if r.get('status') == 'pulado')
err_count  = sum(1 for r in results if r.get('status') not in ('ok','pulado'))

print("\n==================== RESUMO ====================")
print(f"Tasks OK criadas: {ok_count}")
print(f"Pulados (<2 imgs): {skip_count}")
print(f"Erros: {err_count}")
for r in results:
    print(f" - {r['municipio']}: {r.get('status')} ({r.get('msg','')})")

print("\nDica: monitore as tasks com:")
print("for t in ee.batch.Task.list():\n    print(t.status())")
